In [ ]:
# Script plots ERA5 extreme heat season length and change. It can be used to re-create ERA5 figures in the manuscript.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import os
import re
import pandas as pd
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
import regionmask


In [ ]:
# Load in the ERA5 Heat Season Characteristics Files for the two time periods of interest.
# Script is designed to work with one temperature variable (TMAX or TMIN) at a time.

ds1 = xr.load_dataset("/...baseline period TMAX file...")
ds2 = xr.load_dataset("/...comparison period TMAX file...")



In [ ]:
# Grab the array of analytical year start months (see Methods in the paper).
trop_map_p1 = ds1['ref_trop_start_month']

In [ ]:
# PLOT THE AVERAGE LENGTH OF THE EXTREME HEAT SEASON. WILL NEED TO ADJUST TIME PERIOD, TITLES, ETC. DEPENDING ON OBJECTIVE

In [ ]:
def plot_average_season_length(data, title="Average Extreme Heat Season Length"):
    """

    """
    print("Calculating average season length...")

    # 1. Calculate Average (Mean over Years)
    avg_length = data['season_length'].mean(dim='year')
    

    # -------------------------------------------------------------------------
    # PLOTTING
    # -------------------------------------------------------------------------
    print(f"Generating plot: {title}...")
    
    fig = plt.figure(figsize=(12, 7))
    #ax = plt.axes(projection=ccrs.Robinson())
    ax = plt.axes(projection=ccrs.PlateCarree())

    # Previously I masked the tropics for some analysis. This is a legacy of that. Since we aren't masking, this just renames avg_legnth to masked_data.
    # It doesn't do any masking. Keeping this naming convention regardless.
    masked_data = avg_length

    
    custom_bins = [0, 20, 40, 60, 80, 100, 120, 160, 200, 240, 280]

    
    # Plot Data
    p = masked_data.plot(
        ax=ax, transform=ccrs.PlateCarree(),
        cmap='YlOrRd',
        #cmap='Reds',
        #cmap='Oranges',
        #vmin=0, vmax=150, # Force scale to start at 0 days
        #levels=custom_bins, # <-- Pass the custom list here
        #levels=np.arange(0, 180, 20),
        #levels=np.arange(0, 360, 20),
        levels=np.arange(0, 260, 20),
       # cbar_kwargs={'label': 'Days', 'shrink': 0.8},
        cbar_kwargs={'orientation': 'horizontal','label': 'Days', 'aspect': 30, 'pad': 0.08},
        robust=True, 
        zorder=1
    )

    # Mask Ocean (White)
    ax.add_feature(cfeature.OCEAN, color='white', zorder=2)

    # Decoration
    ax.coastlines(linewidth=0.8, color='black', zorder=3)
    ax.add_feature(cfeature.COASTLINE, linestyle=':', alpha=0.5, zorder=3)
    #ax.hlines([-20, 20], -180, 180, 'k', '--', lw=0.5, transform=ccrs.PlateCarree(), zorder=3)

    # Gridlines 
    # Set zorder to 4 so they appear over the white ocean mask
    gl = ax.gridlines(
        draw_labels=True, 
        xlocs=np.arange(-180, 181, 60), 
        ylocs=np.arange(-90, 91, 30), 
        color='lightgray', 
        linewidth=0.8, 
        linestyle='-', 
        zorder=4
    )
    # Turn off labels on the top and right sides for a cleaner look
    gl.top_labels = False
    gl.right_labels = False
    # ----------------------

    plt.title(title, fontsize=14)

    #plt.savefig("plots/ERA5_TMAX_SeasonLength_1966-1995.pdf", format="pdf", bbox_inches="tight")
    #plt.savefig("plots/ERA5_TMAX_SeasonLength_1996-2025.pdf", format="pdf", bbox_inches="tight")


    #plt.savefig("plots/ERA5_TMIN_SeasonLength_1966-1995.pdf", format="pdf", bbox_inches="tight")
    #plt.savefig("plots/ERA5_TMIN_SeasonLength_1996-2025.pdf", format="pdf", bbox_inches="tight")

    
    plt.show()



In [ ]:
# Choose which dataset you are going to plot

#plot_average_season_length(ds1, title="Average TMAX (1966-1995) Extreme Heat Season Length Using 97th Percentile")
plot_average_season_length(ds2, title="Average TMAX (1996-2025) Extreme Heat Season Length Using 97th Percentile")

#plot_average_season_length(ds1, title="Average TMIN (1966-1995) Extreme Heat Season Length Using 97th Percentile")
#plot_average_season_length(ds2, title="Average TMIN (1996-2025) Extreme Heat Season Length Using 97th Percentile")

In [ ]:
# DO SIGNIFICANCE CALCULATION ON THE CHANGE IN HEAT SEASON LENGTH AND THEN PLOT

In [ ]:
def linear_permutation_test(da_p1, da_p2, n_permutations=1000):
    """
    Performs a standard permutation test on LINEAR data (Seasonal Length).
    Tests if the mean duration of P2 is significantly different from P1.
    
    Args:
        da_p1 (xr.DataArray): Time series for Period 1 (lat x lon x year)
        da_p2 (xr.DataArray): Time series for Period 2 (lat x lon x year)
        n_permutations (int): Number of shuffles (default 1000)
        
    Returns:
        p_value_map (xr.DataArray): Map of p-values (0.0 to 1.0)
        obs_diff (xr.DataArray): The observed change (Mean P2 - Mean P1)
    """
    print(f"Starting Linear Permutation Test ({n_permutations} permutations)...")
    
    # --- 1. PREPARE DATA ---
    # Combine data along the 'year' dimension for shuffling
    combined = xr.concat([da_p1, da_p2], dim='year')
    n_p1 = da_p1.sizes['year']
    n_total = combined.sizes['year']
    
    # Calculate Observed Difference (Mean P2 - Mean P1)
    mean_p1 = da_p1.mean(dim='year')
    mean_p2 = da_p2.mean(dim='year')
    obs_diff = mean_p2 - mean_p1
    
    # The test statistic is the absolute magnitude of the change
    obs_stat = np.abs(obs_diff)

    # --- 2. SETUP NUMPY ARRAYS ---
    # Convert to numpy for speed (50x faster than looping xarray)
    # We transpose to ensure 'year' is the LAST axis (axis=-1)
    combined_np = combined.transpose(..., 'year').values
    
    # Initialize counter
    count_larger = np.zeros(combined_np.shape[:-1]) # Shape is (lat, lon)
    
    # Index array for shuffling
    indices = np.arange(n_total)
    
    # --- 3. PERMUTATION LOOP ---
    for i in range(n_permutations):
        if i % 100 == 0: print(f"  Permutation {i}/{n_permutations}...", end='\r')
        
        # Shuffle indices
        np.random.shuffle(indices)
        
        # Split shuffled data indices
        idx_p1 = indices[:n_p1]
        idx_p2 = indices[n_p1:]
        
        # Slice the numpy array
        # We take elements along the last axis (year) using the shuffled indices
        group_p1 = np.take(combined_np, idx_p1, axis=-1)
        group_p2 = np.take(combined_np, idx_p2, axis=-1)
        
        # Calculate Shuffled Means
        # axis=-1 is the year dimension
        m_p1 = np.nanmean(group_p1, axis=-1)
        m_p2 = np.nanmean(group_p2, axis=-1)
        
        # Calculate Shuffled Difference
        perm_diff = m_p2 - m_p1
        
        # Check Magnitude
        # Is the random difference larger than the observed difference?
        count_larger += (np.abs(perm_diff) >= obs_stat.values)

    print(f"  Permutation {n_permutations}/{n_permutations} Complete.")

    # --- 4. FORMAT OUTPUT ---
    # Calculate P-Value: (Count + 1) / (Permutations + 1)
    p_values_np = (count_larger + 1) / (n_permutations + 1)
    
    # Wrap back into Xarray DataArray
    p_value_map = xr.DataArray(
        p_values_np,
        coords=obs_diff.coords,
        dims=obs_diff.dims,
        name='p_value_length'
    )
    
    return p_value_map, obs_diff

In [ ]:
season_lengths_p1 = ds1['season_length']
season_lengths_p2 = ds2['season_length']

# Ensure 'year' is the last dimension for the numpy optimization
# (The script above assumes axis=-1 for year)
season_lengths_p1 = season_lengths_p1.transpose(..., 'year')
season_lengths_p2 = season_lengths_p2.transpose(..., 'year')

In [ ]:
# ==========================================
#  CALCULATE SIGNIFICANCE (P-VALUES)
# ==========================================

# IMPORTANT: CALL THIS NP.RANDOM.SEED(42) COMMAND EVERY TIME YOU RUN A SIGNIFICANCE TEST THAT USES NP.RANDOM.SHUFFLE
# TO ENSURE THE PERMUTATION SHUFFLES ARE EXACTLY REPRODUCIBLE.

np.random.seed(42)

p_val_len, diff_len = linear_permutation_test(season_lengths_p1, season_lengths_p2, n_permutations=1000)

In [ ]:
# AT THIS POINT WE HAVE TESTED SIGNIFICANCE OF THE START AND END DATE CHANGES
# NOW WE RUN THE FDR PROCEDURE

In [ ]:
# FDR Analysis

In [ ]:
def apply_fdr_control(p_map, alpha=0.1, method='indep'):
    """
    Applies the False Discovery Rate (FDR) control to a map of p-values
    following the Benjamini-Hochberg procedure (Wilks, 2016).
    
    Parameters:
    - p_map: xarray DataArray of p-values (lat x lon)
    - alpha: The global significance level (e.g., 0.05)
    - method: 'indep' (assuming independence/weak correlation) or 'dep' (strong correlation)
              Wilks (2016) suggests 'indep' is usually sufficient for climate data.
              
    Returns:
    - sig_mask: Boolean xarray (True where significant)
    """
    print(f"Applying FDR Control (alpha={alpha})...")
    
    # 1. Flatten the map to 1D array
    # We must mask NaNs (ocean/missing data) so they don't count towards N
    p_values = p_map.values.flatten()
    valid_mask = ~np.isnan(p_values)
    p_valid = p_values[valid_mask]
    
    N = len(p_valid) # Total number of tests
    
    # 2. Sort P-values (smallest to largest)
    sorted_p = np.sort(p_valid)
    
    # 3. Calculate Critical Values
    # Formula: (k / N) * alpha
    # k is 1-based index (1, 2, ..., N)
    k = np.arange(1, N + 1)
    
    if method == 'dep':
        # Benjamini-Yekutieli (for strong negative correlations)
        # Usually too conservative for spatial fields
        c_N = np.sum(1 / k)
        p_crit = (k / (N * c_N)) * alpha
    else:
        # Benjamini-Hochberg (Standard)
        p_crit = (k / N) * alpha
        
    # 4. Find Cutoff
    # Find the largest k where p_sorted[k] <= p_crit[k]
    # We check the condition (p <= crit)
    is_below = sorted_p <= p_crit
    
    if np.any(is_below):
        # The largest k is the last True value in the sorted array
        # We find the max index where this is true
        max_k_idx = np.where(is_below)[0].max()
        p_threshold = sorted_p[max_k_idx]
        
        print(f"  > FDR Threshold found: p <= {p_threshold:.5f}")
        print(f"  > (Standard p=0.05 threshold would be less strict)")
    else:
        print("  > No p-values passed FDR control.")
        p_threshold = 0.0
        
    # 5. Create Significance Mask
    sig_mask = p_map <= p_threshold
    
    return sig_mask

In [ ]:
# --- PREP DATA FOR MASKING ---
def shift_lon_and_clean(da):
    """Aligns longitudes to -180 to 180 and removes overlapping seams."""
    da_shifted = da.assign_coords(lon=(((da.lon + 180) % 360) - 180)).sortby('lon')
    return da_shifted.drop_duplicates(dim='lon')

print("Realigning longitudes and removing duplicate seams...")
# Shift both the p-values and the difference map before masking
p_val_len = shift_lon_and_clean(p_val_len)
diff_len = shift_lon_and_clean(diff_len)


# --- Apply FDR Control ---
print("Masking oceans prior to FDR control...")
# 1. Generate the land mask (it will now succeed because duplicates are gone!)
land_mask = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(p_val_len)

# 2. Mask out the ocean p-values so they don't count toward 'N' in the FDR test
p_val_len_land_only = p_val_len.where(land_mask == 0)

# 3. Apply FDR ONLY to the land pixels
sig_len_mask = apply_fdr_control(p_val_len_land_only, alpha=0.05)

In [ ]:
# NOW PLOT THE DIFFERENCES

In [ ]:
def plot_length_change_fdr(differences, sig_mask, title, label='Days', cmap='RdBu_r'):
    """
    Plots the change for a single variable, masking non-significant pixels
    and masking the ocean. Matches the exact formatting of the percent change plot.
    """
    print("1. Applying FDR Mask...")
    # 1. Apply FDR Mask
    diff_masked = differences.where(sig_mask)

    print("2. Generating Map...")
    # 3. Plotting
    fig = plt.figure(figsize=(12, 7))
    ax = plt.axes(projection=ccrs.PlateCarree())

    #custom_bins = [-180, -160, -140, -120, -100, -80, -60, -50, -40, -30, -20, -10, 0, 10, 20, 30, 40, 50, 60, 80, 100, 120, 140, 160, 180]

    # Plot Data (zorder=1)
    diff_masked.plot(
        ax=ax, 
        transform=ccrs.PlateCarree(),
        cmap=cmap, 
        #levels=custom_bins,
        levels=np.arange(-120, 130, 10),
        extend='both',
        cbar_kwargs={
            'orientation': 'horizontal',
            'label': label, 
            'aspect': 30, 
            'pad': 0.08,
            'ticks': np.arange(-120, 121, 40)
        },
        zorder=1
    )
    
    # --- MASK OCEAN (zorder=2) ---
    ax.add_feature(cfeature.OCEAN, color='white', zorder=2)

    # Decoration (zorder=3)
    ax.coastlines(color='black', linewidth=0.8, zorder=3)
    ax.add_feature(cfeature.COASTLINE, linestyle=':', alpha=0.5, zorder=3)

    # --- 5. GRIDLINES (zorder=4) ---
    # Initializing at the absolute end forces them on top of the ocean polygon layer
    gl = ax.gridlines(
        draw_labels=True, 
        xlocs=np.arange(-180, 181, 60), 
        ylocs=np.arange(-90, 91, 30), 
        color='lightgray', 
        linewidth=0.8, 
        linestyle='-', 
        zorder=4
    )
    
    # This forces the labels to only appear on the bottom and left
    gl.top_labels = False
    gl.right_labels = False
    
    gl.zorder = 4

    plt.title(title, fontsize=14)
   # plt.savefig("plots/ERA5_TMAX_SeasonLength_Change.pdf", format="pdf", bbox_inches="tight")
    #plt.savefig("plots/ERA5_TMIN_SeasonLength_Change.pdf", format="pdf", bbox_inches="tight")
    
    # Force the map's boundary box to render on top of the gridlines
    ax.spines['geo'].set_zorder(5)
    plt.show()
    
    return diff_masked

In [ ]:
plot_length_change_fdr(
    diff_len, 
    sig_len_mask, 
    title="Change (1996-2025) - (1966-1995) in TMAX Extreme Heat Season Length",
    #title="Change (1996-2025) - (1966-1995) in TMIN Extreme Heat Season Length",
    label="Days",
    cmap="RdBu_r" # Red = Longer Season
);

In [ ]:
def plot_percent_length_change_fdr(differences, reference_length, sig_mask, title, label='Percent Change (%)', cmap='RdBu_r'):
    """
    Calculates percent change, masks non-significant pixels using an FDR mask,
    masks the ocean.
    """
    print("1. Preparing data and calculating percentages...")
    
    # If the reference data still has a 'year' dimension, average it out
    if 'year' in reference_length.dims:
        print("   -> Averaging 'reference_length' over the 'year' dimension...")
        ref_2d = reference_length.mean(dim='year', skipna=True)
    else:
        ref_2d = reference_length
        
    # --- Align reference longitudes to match 'differences' ---
    print("   -> Realigning reference longitudes (-180 to 180)...")
    ref_2d = ref_2d.assign_coords(lon=(((ref_2d.lon + 180) % 360) - 180)).sortby('lon')
    ref_2d = ref_2d.drop_duplicates(dim='lon')
    # ------------------------------------------------------------------
        
    # Only divide where ref_2d > 0 to avoid dividing by zero
    percent_change = xr.where(
        ref_2d > 0, 
        (differences / ref_2d) * 100, 
        np.nan
    )

    # 2. Apply FDR Mask
    percent_masked = percent_change.where(sig_mask)

    print("2. Generating Map...")
    fig = plt.figure(figsize=(12, 7))
    ax = plt.axes(projection=ccrs.PlateCarree())

    # Plot Data (zorder=1)
    percent_masked.plot(
        ax=ax, 
        transform=ccrs.PlateCarree(),
        cmap=cmap, 
        levels=np.arange(-150, 165, 15), 
        extend='both',
        cbar_kwargs={
            'orientation': 'horizontal', 
            'label': label, 
            'aspect': 30, 
            'pad': 0.08,
            'ticks': np.arange(-150, 151, 30) 
        },
        zorder=1
    )
    
    # --- MASK OCEAN (zorder=2) ---
    ax.add_feature(cfeature.OCEAN, color='white', zorder=2)
    
    # Decoration (zorder=3)
    ax.coastlines(color='black', linewidth=0.8, zorder=3)
    ax.add_feature(cfeature.COASTLINE, linestyle=':', alpha=0.5, zorder=3)

    # --- 5. GRIDLINES (zorder=4) ---
    gl = ax.gridlines(
        draw_labels=True, 
        xlocs=np.arange(-180, 181, 60), 
        ylocs=np.arange(-90, 91, 30), 
        color='lightgray', 
        linewidth=0.8, 
        linestyle='-', 
        zorder=4
    )

    gl.top_labels = False
    gl.right_labels = False
        
    gl.zorder = 4
    
    plt.title(title, fontsize=14)
    #plt.savefig("plots/ERA5_TMAX_SeasonLength_PercentChange.pdf", format="pdf", bbox_inches="tight")
    
    ax.spines['geo'].set_zorder(5)
    plt.show()
    
    return percent_change, percent_masked

In [ ]:
percent_map, masked_percent_map = plot_percent_length_change_fdr(
    diff_len, 
    season_lengths_p1, 
    sig_len_mask, # <--- Pass your boolean FDR mask here
    title="Percent Change (1996-2025) - (1966-1995)in TMAX Extreme Heat Season Length"
    #title="Percent Change (1996-2025) - (1966-1995)in TMIN Extreme Heat Season Length"
)

In [ ]:
# This code was used to calcuate fractions of land area for a given change presented in the manuscript text


# ==========================================
#  CALCULATE SIGNIFICANT LAND AREA FRACTIONS
# ==========================================
print("\nCalculating land area fractions...")

# 1. Create a true data mask for the land (since cfeature.OCEAN is only a visual overlay)
#land_mask = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(diff_len)
is_land = (land_mask == 0)

# 2. Create Area Weights (Cosine of latitude)
weights = np.cos(np.deg2rad(diff_len.lat))
land_weights = weights * xr.ones_like(diff_len).where(is_land)

# 3. Calculate Total Land Area (sum of weights)
total_land_area = land_weights.sum().compute().item()

# 4. Isolate Significant Increases and Decreases (over land only)
sig_increase = sig_len_mask & (diff_len > 0)
sig_decrease = sig_len_mask & (diff_len < 0)

# 5. Sum the weights for the significant regions
area_increase = land_weights.where(sig_increase).sum().compute().item()
area_decrease = land_weights.where(sig_decrease).sum().compute().item()

# 6. Calculate Percentages
pct_increase = (area_increase / total_land_area) * 100
pct_decrease = (area_decrease / total_land_area) * 100
pct_none = 100 - (pct_increase + pct_decrease)

print("\n" + "="*50)
print(" LAND AREA SIGNIFICANCE SUMMARY (1996-2025 vs 1966-1995)")
print("="*50)
print(f" Statistically Significant Increase: {pct_increase:>5.1f}% of global land")
print(f" Statistically Significant Decrease: {pct_decrease:>5.1f}% of global land")
print(f" No Significant Change:              {pct_none:>5.1f}% of global land")
print("="*50 + "\n")

In [ ]:
# This code was used to calcuate fractions of land area for a given change presented in the manuscript text

# ==========================================
#  UNMASKED REGIONAL AVERAGES (Area-Weighted)
# ==========================================
print("\n" + "="*55)
print(" UNMASKED REGIONAL AVERAGE CHANGES (Area-Weighted Land)")
print("="*55)

# 1. Isolate land-only pixels using the raw, unmasked differences
# (Assumes land_mask was generated via regionmask earlier in the script)
diff_land = diff_len.where(land_mask == 0)

# 2. Create area weights (cosine of latitude)
weights = np.cos(np.deg2rad(diff_land.lat))

# Helper function to calculate the weighted mean for a given data slice
def get_weighted_mean(da_region):
    # xarray's built-in weighted mean automatically handles NaNs
    return da_region.weighted(weights).mean(dim=['lat', 'lon'], skipna=True).compute().item()

# 3. Define the regional subsets based on absolute latitude
global_da   = diff_land
tropical_da = diff_land.where(abs(diff_land.lat) <= 23.5)
midlat_da   = diff_land.where((abs(diff_land.lat) > 23.5) & (abs(diff_land.lat) <= 66.5))
highlat_da  = diff_land.where(abs(diff_land.lat) > 66.5)

# 4. Calculate and print the regional averages
print(f" Global Land Average:        {get_weighted_mean(global_da):>6.2f} days")
print(f" Tropical Land Average:      {get_weighted_mean(tropical_da):>6.2f} days (|lat| <= 23.5)")
print(f" Midlatitude Land Average:   {get_weighted_mean(midlat_da):>6.2f} days (23.5 < |lat| <= 66.5)")
print(f" High Latitude Land Average: {get_weighted_mean(highlat_da):>6.2f} days (|lat| > 66.5)")
print("="*55 + "\n")

In [ ]:
# This code was used to calcuate fractions of land area for a given change presented in the manuscript text

# ==========================================
#  UNMASKED REGIONAL PERCENT CHANGES (Area-Weighted)
# ==========================================
print("\n" + "="*55)
print(" UNMASKED REGIONAL PERCENT CHANGES (Area-Weighted Land)")
print("="*55)

# 1. Calculate the baseline mean (Period 1)
mean_p1_raw = season_lengths_p1.mean(dim='year', skipna=True)

# ---> THE FIX: Shift the baseline mean to match diff_len (-180 to 180)
mean_p1 = shift_lon_and_clean(mean_p1_raw)

# 2. Calculate percent change
# Use xr.where to avoid dividing by 0 (which creates 'inf' and breaks the average)
pct_change = xr.where(mean_p1 == 0, np.nan, (diff_len / mean_p1) * 100)

# 3. Isolate land-only pixels
pct_land = pct_change.where(land_mask == 0)

# 4. Create area weights (cosine of latitude)
weights = np.cos(np.deg2rad(pct_land.lat))

# Helper function for weighted mean
def get_weighted_mean_pct(da_region):
    return da_region.weighted(weights).mean(dim=['lat', 'lon'], skipna=True).compute().item()

# 5. Define the regional subsets
global_pct   = pct_land
tropical_pct = pct_land.where(abs(pct_land.lat) <= 23.5)
midlat_pct   = pct_land.where((abs(pct_land.lat) > 23.5) & (abs(pct_land.lat) <= 66.5))
highlat_pct  = pct_land.where(abs(pct_land.lat) > 66.5)

# 6. Calculate and print the regional averages
print(f" Global Land Average:        {get_weighted_mean_pct(global_pct):>7.2f}%")
print(f" Tropical Land Average:      {get_weighted_mean_pct(tropical_pct):>7.2f}% (|lat| <= 23.5)")
print(f" Midlatitude Land Average:   {get_weighted_mean_pct(midlat_pct):>7.2f}% (23.5 < |lat| <= 66.5)")
print(f" High Latitude Land Average: {get_weighted_mean_pct(highlat_pct):>7.2f}% (|lat| > 66.5)")
print("="*55 + "\n")